In [1]:
import pickle
from pathlib import Path
import numpy as np
import scipy as sp
import os

import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import random

from tqdm import tqdm

gpu = "0"
device = torch.device(f"cuda:{gpu}" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
batch_size = 64
dropout_mlp = 0.5
dropout_gru = 0.25
learning_rate = 1e-4
weight_decay = 1e-2

Using device: cpu


In [ ]:
inference_results = list(Path("./results/800").rglob("*.pickle"))
print(inference_results)
os.makedirs("./models/", exist_ok=True)

[WindowsPath('results/800/gpt2_trivia_qa_start-0_end-800.pickle')]


In [ ]:
class FFHallucinationClassifier(torch.nn.Module):
    def __init__(self, input_shape, dropout = dropout_mlp):
        super().__init__()
        self.dropout = dropout
        
        self.linear_relu_stack =torch.nn.Sequential(
            torch.nn.Linear(input_shape, 256),
            torch.nn.ReLU(),
            torch.nn.Dropout(self.dropout),
            torch.nn.Linear(256, 2)
            )

    def forward(self, x):
        logits = self.linear_relu_stack(x)
        return logits

class RNNHallucinationClassifier(torch.nn.Module):
    def __init__(self, dropout=dropout_gru):
        super().__init__()
        hidden_dim = 128
        num_layers = 4
        self.gru = torch.nn.GRU(1, hidden_dim, num_layers, dropout=dropout, batch_first=True, bidirectional=False)
        self.linear = torch.nn.Linear(hidden_dim, 2)
    
    def forward(self, seq):
        gru_out, _ = self.gru(seq)
        return self.linear(gru_out)[-1, -1, :]

In [ ]:
def gen_classifier_roc(inputs, save_name="mlp_model.pt"):
    X_train, X_test, y_train, y_test = train_test_split(inputs, correct.astype(int), test_size=0.2, random_state=123)
    classifier_model = FFHallucinationClassifier(X_train.shape[1]).to(device)
    X_train = torch.tensor(X_train).to(device)
    y_train = torch.tensor(y_train).to(torch.long).to(device)
    X_test = torch.tensor(X_test).to(device)
    y_test = torch.tensor(y_test).to(torch.long).to(device)

    optimizer = torch.optim.AdamW(classifier_model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    for _ in tqdm(range(1001)):
        optimizer.zero_grad()
        sample = torch.randperm(X_train.shape[0])[:batch_size]
        pred = classifier_model(X_train[sample])
        loss = torch.nn.functional.cross_entropy(pred, y_train[sample])
        loss.backward()
        optimizer.step()
    torch.save(classifier_model.state_dict(), f"./models/{save_name}")
    classifier_model.eval()
    with torch.no_grad():
        pred = torch.nn.functional.softmax(classifier_model(X_test), dim=1)
        prediction_classes = (pred[:,1]>0.5).type(torch.long).cpu()
        return roc_auc_score(y_test.cpu(), pred[:,1].cpu()), (prediction_classes.numpy()==y_test.cpu().numpy()).mean()

In [5]:
all_results = {}

In [7]:
for results_file in inference_results:
    if results_file not in all_results.keys():
        try:
            del results
        except:
            pass
        try:
            classifier_results = {}
            with open(results_file, "rb") as infile:
                results = pickle.loads(infile.read())
            correct = np.array(results['correct'])
    
            # attributes
            X_train, X_test, y_train, y_test = train_test_split(results['attributes_first'], correct.astype(int), test_size=0.2, random_state=1234)
            rnn_model = RNNHallucinationClassifier()
            optimizer = torch.optim.AdamW(rnn_model.parameters(), lr=learning_rate, weight_decay=weight_decay)
            for step in tqdm(range(1001)):
                x_sub, y_sub = zip(*random.sample(list(zip(X_train, y_train)), batch_size))
                y_sub = torch.tensor(y_sub).to(torch.long)
                optimizer.zero_grad()
                preds = torch.stack([rnn_model(torch.tensor(i).view(1, -1, 1).to(torch.float)) for i in x_sub])
                loss = torch.nn.functional.cross_entropy(preds, y_sub)
                loss.backward()
                optimizer.step()
            torch.save(rnn_model.state_dict(), f"rnn_model.pt")
            
            preds = torch.stack([rnn_model(torch.tensor(i).view(1, -1, 1).to(torch.float)) for i in X_test])
            preds = torch.nn.functional.softmax(preds, dim=1)
            prediction_classes = (preds[:,1]>0.5).type(torch.long).cpu()
            classifier_results['attribution_rnn_roc'] = roc_auc_score(y_test, preds[:,1].detach().cpu().numpy())
            classifier_results['attribution_rnn_acc'] = (prediction_classes.numpy()==y_test).mean()

            del rnn_model, preds, prediction_classes, X_train, X_test, y_train, y_test, results['attributes_first']

            # logits
            first_logits = np.stack([sp.special.softmax(i[j]) for i, j in zip(results['logits'], results['start_pos'])])
            first_logits_roc, first_logits_acc = gen_classifier_roc(first_logits, save_name="mlp_logit.pt")
            classifier_results['first_logits_roc'] = first_logits_roc
            classifier_results['first_logits_acc'] = first_logits_acc

            del first_logits

            # fully connected
            for layer in range(results['first_fully_connected'][0].shape[0]):
                layer_roc, layer_acc = gen_classifier_roc(np.stack([i[layer] for i in results['first_fully_connected']]), save_name=f"mlp_fc_{layer}.pt")
                classifier_results[f'first_fully_connected_roc_{layer}'] = layer_roc
                classifier_results[f'first_fully_connected_acc_{layer}'] = layer_acc
            
            del results['first_fully_connected']

            # attention
            for layer in range(results['first_attention'][0].shape[0]):
                layer_roc, layer_acc = gen_classifier_roc(np.stack([i[layer] for i in results['first_attention']]), save_name=f"mlp_attn_{layer}.pt")
                classifier_results[f'first_attention_roc_{layer}'] = layer_roc
                classifier_results[f'first_attention_acc_{layer}'] = layer_acc
            
            del results['first_attention']
            
            all_results[results_file] = classifier_results.copy()
        except Exception as err:
            print(err)

100%|██████████| 1001/1001 [00:02<00:00, 400.20it/s]


In [10]:
print(all_results.keys())

dict_keys([WindowsPath('results/800/gpt2_trivia_qa_start-0_end-800.pickle')])


In [12]:
for k,v in all_results.items():
    for kk, vv in v.items():
        print(kk, vv)

attribution_rnn_roc 0.6173333333333333
attribution_rnn_acc 0.9375
first_logits_roc 0.6496815286624203
first_logits_acc 0.98125
first_fully_connected_roc_0 0.5074309978768577
first_fully_connected_acc_0 0.98125
first_fully_connected_roc_1 0.4394904458598726
first_fully_connected_acc_1 0.98125
first_fully_connected_roc_2 0.5095541401273885
first_fully_connected_acc_2 0.98125
first_fully_connected_roc_3 0.7303609341825902
first_fully_connected_acc_3 0.98125
first_fully_connected_roc_4 0.613588110403397
first_fully_connected_acc_4 0.98125
first_fully_connected_roc_5 0.5456475583864119
first_fully_connected_acc_5 0.96875
first_fully_connected_roc_6 0.5690021231422505
first_fully_connected_acc_6 0.98125
first_fully_connected_roc_7 0.6199575371549894
first_fully_connected_acc_7 0.98125
first_fully_connected_roc_8 0.6900212314225053
first_fully_connected_acc_8 0.975
first_fully_connected_roc_9 0.6114649681528662
first_fully_connected_acc_9 0.98125
first_fully_connected_roc_10 0.607218683651804

AUROC on GPT-2 and TriviaQA (800 samples):
- IG attribution: 0.62
- Softmax probability: 0.65
- Fully-connected activation: 0.78 (layer 11)
- Self-attention score: 0.72 (layer 7) | 0.62 (layer 11)

AUROC on GPT-2 and TriviaQA (250 samples):
- IG attribution: 0.67
- Softmax probability: 0.36
- Fully-connected activation: 0.67 (layer 0) | 0.51 (layer 11)
- Self-attention score: 0.67 (layer 5) | 0.36 (layer 11)

### Merge multiple pickle files

In [ ]:
import pickle
import glob
from collections import defaultdict
from tqdm import tqdm
import os

results_dir = "./results/pv_v1"
pattern = os.path.join(results_dir, "*_batch_*.pickle")

batch_files = sorted(glob.glob(pattern))
if not batch_files:
    raise FileNotFoundError("No batch pickle files found.")

print(f"Found {len(batch_files)} batch files.")

# Initialize empty lists
combined = defaultdict(list)

# Iterate over each batch file and extend lists
for bf in tqdm(batch_files, desc="Merging batches"):
    with open(bf, "rb") as f:
        batch_data = pickle.load(f)
    for key, values in batch_data.items():
        combined[key].extend(values)
        
output_file = os.path.join(results_dir, "gpt2_birth_qa_start-0_end-500_pv.pickle")
with open(output_file, "wb") as f:
    pickle.dump(dict(combined), f)

print(f"Combined results saved to {output_file}")

Found 5 batch files.


Merging batches: 100%|██████████| 5/5 [00:06<00:00,  1.24s/it]


Combined results saved to ./results/pv_v1\gpt2_birth_qa_start-0_end-800_pv.pickle


### Eval on TRex

In [ ]:
def eval_mlp_logit(inputs):
    _, X_test, _, y_test = train_test_split(inputs, correct.astype(int), test_size=0.9, random_state=123)
    classifier_model = FFHallucinationClassifier(X_test.shape[1]).to(device)
    classifier_model.load_state_dict(torch.load("models/mlp_logit.pt", map_location=device))
    X_test = torch.tensor(X_test).to(device)
    y_test = torch.tensor(y_test).to(torch.long).to(device)
    classifier_model.eval()
    with torch.no_grad():
        pred = torch.nn.functional.softmax(classifier_model(X_test), dim=1)
        prediction_classes = (pred[:,1]>0.5).type(torch.long).cpu()
        return roc_auc_score(y_test.cpu(), pred[:,1].cpu()), (prediction_classes.numpy()==y_test.cpu().numpy()).mean()
    
def eval_mlp_fc(inputs, layer):
    _, X_test, _, y_test = train_test_split(inputs, correct.astype(int), test_size=0.9, random_state=123)
    classifier_model = FFHallucinationClassifier(X_test.shape[1]).to(device)
    classifier_model.load_state_dict(torch.load(f"models/mlp_fc_{layer}.pt", map_location=device))
    X_test = torch.tensor(X_test).to(device)
    y_test = torch.tensor(y_test).to(torch.long).to(device)
    classifier_model.eval()
    with torch.no_grad():
        pred = torch.nn.functional.softmax(classifier_model(X_test), dim=1)
        prediction_classes = (pred[:,1]>0.5).type(torch.long).cpu()
        return roc_auc_score(y_test.cpu(), pred[:,1].cpu()), (prediction_classes.numpy()==y_test.cpu().numpy()).mean()
    
def eval_mlp_attn(inputs, layer):
    _, X_test, _, y_test = train_test_split(inputs, correct.astype(int), test_size=0.9, random_state=123)
    classifier_model = FFHallucinationClassifier(X_test.shape[1]).to(device)
    classifier_model.load_state_dict(torch.load(f"models/mlp_attn_{layer}.pt", map_location=device))
    X_test = torch.tensor(X_test).to(device)
    y_test = torch.tensor(y_test).to(torch.long).to(device)
    classifier_model.eval()
    with torch.no_grad():
        pred = torch.nn.functional.softmax(classifier_model(X_test), dim=1)
        prediction_classes = (pred[:,1]>0.5).type(torch.long).cpu()
        return roc_auc_score(y_test.cpu(), pred[:,1].cpu()), (prediction_classes.numpy()==y_test.cpu().numpy()).mean()

In [ ]:
inference_results = list(Path("./results/trex").rglob("*.pickle"))
print(inference_results)
for results_file in inference_results:
    if results_file not in all_results.keys():
        try:
            del results
        except:
            pass
        try:
            classifier_results = {}
            with open(results_file, "rb") as infile:
                results = pickle.loads(infile.read())
            correct = np.array(results['correct'])
    
            # attributes
            X_train, X_test, y_train, y_test = train_test_split(results['attributes_first'], correct.astype(int), test_size=0.9, random_state=1234)
            rnn_model = RNNHallucinationClassifier().to(device)
            rnn_model.load_state_dict(torch.load("models/rnn_model.pt", map_location=device))
            rnn_model.eval()
            # optimizer = torch.optim.AdamW(rnn_model.parameters(), lr=learning_rate, weight_decay=weight_decay)
            # for step in tqdm(range(1001)):
            #     x_sub, y_sub = zip(*random.sample(list(zip(X_train, y_train)), batch_size))
            #     y_sub = torch.tensor(y_sub).to(torch.long)
            #     optimizer.zero_grad()
            #     preds = torch.stack([rnn_model(torch.tensor(i).view(1, -1, 1).to(torch.float)) for i in x_sub])
            #     loss = torch.nn.functional.cross_entropy(preds, y_sub)
            #     loss.backward()
            #     optimizer.step()
            # torch.save(rnn_model.state_dict(), f"rnn_model.pt")
            
            preds = torch.stack([rnn_model(torch.tensor(i).view(1, -1, 1).to(torch.float)) for i in X_test])
            preds = torch.nn.functional.softmax(preds, dim=1)
            prediction_classes = (preds[:,1]>0.5).type(torch.long).cpu()
            classifier_results['attribution_rnn_roc'] = roc_auc_score(y_test, preds[:,1].detach().cpu().numpy())
            classifier_results['attribution_rnn_acc'] = (prediction_classes.numpy()==y_test).mean()

            # logits
            first_logits = np.stack([sp.special.softmax(i[j]) for i, j in zip(results['logits'], results['start_pos'])])
            first_logits_roc, first_logits_acc = eval_mlp_logit(first_logits)
            classifier_results['first_logits_roc'] = first_logits_roc
            classifier_results['first_logits_acc'] = first_logits_acc

            # fully connected
            for layer in range(results['first_fully_connected'][0].shape[0]):
                layer_roc, layer_acc = eval_mlp_fc(np.stack([i[layer] for i in results['first_fully_connected']]), layer)
                classifier_results[f'first_fully_connected_roc_{layer}'] = layer_roc
                classifier_results[f'first_fully_connected_acc_{layer}'] = layer_acc

            # attention
            for layer in range(results['first_attention'][0].shape[0]):
                layer_roc, layer_acc = eval_mlp_attn(np.stack([i[layer] for i in results['first_attention']]), layer)
                classifier_results[f'first_attention_roc_{layer}'] = layer_roc
                classifier_results[f'first_attention_acc_{layer}'] = layer_acc
            
            all_results[results_file] = classifier_results.copy()
        except Exception as err:
            print(err)

[WindowsPath('results/trex/gpt2_birth_qa_start-0_end-160.pickle'), WindowsPath('results/trex/gpt2_capitals_qa_start-0_end-160.pickle'), WindowsPath('results/trex/gpt2_founders_qa_start-0_end-160.pickle')]


In [32]:
for k,v in all_results.items():
    print(f"Results for {k}:")
    for kk, vv in v.items():
        print(kk, vv)

Results for results\800\gpt2_trivia_qa_start-0_end-800.pickle:
first_logits_roc 0.6475583864118896
first_logits_acc 0.98125
first_fully_connected_roc_0 0.49044585987261147
first_fully_connected_acc_0 0.98125
first_fully_connected_roc_1 0.42462845010615713
first_fully_connected_acc_1 0.98125
first_fully_connected_roc_2 0.4840764331210191
first_fully_connected_acc_2 0.98125
first_fully_connected_roc_3 0.721868365180467
first_fully_connected_acc_3 0.975
first_fully_connected_roc_4 0.6220806794055201
first_fully_connected_acc_4 0.98125
first_fully_connected_roc_5 0.5414012738853503
first_fully_connected_acc_5 0.96875
first_fully_connected_roc_6 0.5477707006369427
first_fully_connected_acc_6 0.98125
first_fully_connected_roc_7 0.6220806794055201
first_fully_connected_acc_7 0.975
first_fully_connected_roc_8 0.732484076433121
first_fully_connected_acc_8 0.975
first_fully_connected_roc_9 0.5902335456475584
first_fully_connected_acc_9 0.98125
first_fully_connected_roc_10 0.5881104033970276
firs

ROC on TRex (3 domains):

Birth: 
- IG: 0.69
- Softmax: 0.55
- FC: 0.68 (layer 11)
- Attention: 0.84 (layer 4) | 0.69 (layer 11)

Capitals:
- IG: 0.31
- Softmax: 0.55
- FC: 0.68 (layer 11)
- Attention: 0.58 (layer 8) | 0.42 (layer 11)

Founders:
- IG: 0.35
- Softmax: 0.47
- FC: 0.44 (layer 11)
- Attention: 0.52 (layer 8) | 0.28 (layer 11)

### Implement 2-layer classifier

In [2]:
class FFHallucinationClassifier_2_layer(torch.nn.Module):
    def __init__(self, input_shape, dropout = dropout_mlp):
        super().__init__()
        self.dropout = dropout
        
        self.linear_relu_stack =torch.nn.Sequential(
            # First hidden layer
            torch.nn.Linear(input_shape, 256),
            torch.nn.ReLU(),
            torch.nn.Dropout(self.dropout),

            # Second hidden layer
            torch.nn.Linear(256, 256),
            torch.nn.ReLU(),
            torch.nn.Dropout(self.dropout),

            torch.nn.Linear(256, 2)
            )

    def forward(self, x):
        logits = self.linear_relu_stack(x)
        return logits
    
def gen_classifier_roc_2_layer(inputs):
    X_train, X_test, y_train, y_test = train_test_split(inputs, correct.astype(int), test_size=0.2, random_state=123)
    classifier_model = FFHallucinationClassifier_2_layer(X_train.shape[1]).to(device)
    X_train = torch.tensor(X_train).to(device)
    y_train = torch.tensor(y_train).to(torch.long).to(device)
    X_test = torch.tensor(X_test).to(device)
    y_test = torch.tensor(y_test).to(torch.long).to(device)

    optimizer = torch.optim.AdamW(classifier_model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    for _ in tqdm(range(1001)):
        optimizer.zero_grad()
        sample = torch.randperm(X_train.shape[0])[:batch_size]
        pred = classifier_model(X_train[sample])
        loss = torch.nn.functional.cross_entropy(pred, y_train[sample])
        loss.backward()
        optimizer.step()
    classifier_model.eval()
    with torch.no_grad():
        pred = torch.nn.functional.softmax(classifier_model(X_test), dim=1)
        prediction_classes = (pred[:,1]>0.5).type(torch.long).cpu()
        return roc_auc_score(y_test.cpu(), pred[:,1].cpu()), (prediction_classes.numpy()==y_test.cpu().numpy()).mean()

In [3]:
inference_results = list(Path("./results/800").rglob("*.pickle"))
print(inference_results)
all_results = {}

[WindowsPath('results/800/gpt2_trivia_qa_start-0_end-800.pickle')]


In [4]:
for results_file in inference_results:
    if results_file not in all_results.keys():
        try:
            del results
        except:
            pass
        try:
            classifier_results = {}
            with open(results_file, "rb") as infile:
                results = pickle.loads(infile.read())
            correct = np.array(results['correct'])

            # logits
            first_logits = np.stack([sp.special.softmax(i[j]) for i, j in zip(results['logits'], results['start_pos'])])
            first_logits_roc, first_logits_acc = gen_classifier_roc_2_layer(first_logits)
            classifier_results['first_logits_roc'] = first_logits_roc
            classifier_results['first_logits_acc'] = first_logits_acc

            # fully connected
            for layer in range(results['first_fully_connected'][0].shape[0]):
                layer_roc, layer_acc = gen_classifier_roc_2_layer(np.stack([i[layer] for i in results['first_fully_connected']]))
                classifier_results[f'first_fully_connected_roc_{layer}'] = layer_roc
                classifier_results[f'first_fully_connected_acc_{layer}'] = layer_acc

            # attention
            for layer in range(results['first_attention'][0].shape[0]):
                layer_roc, layer_acc = gen_classifier_roc_2_layer(np.stack([i[layer] for i in results['first_attention']]))
                classifier_results[f'first_attention_roc_{layer}'] = layer_roc
                classifier_results[f'first_attention_acc_{layer}'] = layer_acc
            
            all_results[results_file] = classifier_results.copy()
        except Exception as err:
            print(err)

100%|██████████| 1001/1001 [00:04<00:00, 232.64it/s]


In [5]:
for k,v in all_results.items():
    print(f"Results for {k}:")
    for kk, vv in v.items():
        print(kk, vv)

Results for results\800\gpt2_trivia_qa_start-0_end-800.pickle:
first_logits_roc 0.3503184713375796
first_logits_acc 0.98125
first_fully_connected_roc_0 0.5138004246284501
first_fully_connected_acc_0 0.98125
first_fully_connected_roc_1 0.5031847133757962
first_fully_connected_acc_1 0.98125
first_fully_connected_roc_2 0.5923566878980892
first_fully_connected_acc_2 0.98125
first_fully_connected_roc_3 0.721868365180467
first_fully_connected_acc_3 0.975
first_fully_connected_roc_4 0.6008492569002123
first_fully_connected_acc_4 0.98125
first_fully_connected_roc_5 0.564755838641189
first_fully_connected_acc_5 0.975
first_fully_connected_roc_6 0.5605095541401274
first_fully_connected_acc_6 0.975
first_fully_connected_roc_7 0.6220806794055201
first_fully_connected_acc_7 0.975
first_fully_connected_roc_8 0.6602972399150743
first_fully_connected_acc_8 0.975
first_fully_connected_roc_9 0.6900212314225053
first_fully_connected_acc_9 0.98125
first_fully_connected_roc_10 0.6539278131634819
first_full

AUROC of 2-layer NN on GPT-2 and TriviaQA (800 samples):
- Softmax probability: 0.35
- Fully-connected activation: 0.84 (layer 11)
- Self-attention score: 0.75 (layer 6) | 0.67 (layer 11)

### Implement 5-layer classifier

In [2]:
class FFHallucinationClassifier_5_layer(torch.nn.Module):
    def __init__(self, input_shape, dropout = dropout_mlp):
        super().__init__()
        self.dropout = dropout
        
        self.linear_relu_stack =torch.nn.Sequential(
            # First hidden layer
            torch.nn.Linear(input_shape, 256),
            torch.nn.ReLU(),
            torch.nn.Dropout(self.dropout),

            # Second hidden layer
            torch.nn.Linear(256, 256),
            torch.nn.ReLU(),
            torch.nn.Dropout(self.dropout),

            # Third
            torch.nn.Linear(256, 256),
            torch.nn.ReLU(),
            torch.nn.Dropout(self.dropout),

            # Fourth
            torch.nn.Linear(256, 256),
            torch.nn.ReLU(),
            torch.nn.Dropout(self.dropout),

            # Fifth
            torch.nn.Linear(256, 256),
            torch.nn.ReLU(),
            torch.nn.Dropout(self.dropout),

            torch.nn.Linear(256, 2)
            )

    def forward(self, x):
        logits = self.linear_relu_stack(x)
        return logits
    
def gen_classifier_roc_5_layer(inputs):
    X_train, X_test, y_train, y_test = train_test_split(inputs, correct.astype(int), test_size=0.2, random_state=123)
    classifier_model = FFHallucinationClassifier_5_layer(X_train.shape[1]).to(device)
    X_train = torch.tensor(X_train).to(device)
    y_train = torch.tensor(y_train).to(torch.long).to(device)
    X_test = torch.tensor(X_test).to(device)
    y_test = torch.tensor(y_test).to(torch.long).to(device)

    optimizer = torch.optim.AdamW(classifier_model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    for _ in tqdm(range(1001)):
        optimizer.zero_grad()
        sample = torch.randperm(X_train.shape[0])[:batch_size]
        pred = classifier_model(X_train[sample])
        loss = torch.nn.functional.cross_entropy(pred, y_train[sample])
        loss.backward()
        optimizer.step()
    classifier_model.eval()
    with torch.no_grad():
        pred = torch.nn.functional.softmax(classifier_model(X_test), dim=1)
        prediction_classes = (pred[:,1]>0.5).type(torch.long).cpu()
        return roc_auc_score(y_test.cpu(), pred[:,1].cpu()), (prediction_classes.numpy()==y_test.cpu().numpy()).mean()

In [3]:
inference_results = list(Path("./results/800").rglob("*.pickle"))
print(inference_results)
all_results = {}

[WindowsPath('results/800/gpt2_trivia_qa_start-0_end-800.pickle')]


In [4]:
for results_file in inference_results:
    if results_file not in all_results.keys():
        try:
            del results
        except:
            pass
        try:
            classifier_results = {}
            with open(results_file, "rb") as infile:
                results = pickle.loads(infile.read())
            correct = np.array(results['correct'])

            # logits
            print("Classify on logits")
            first_logits = np.stack([sp.special.softmax(i[j]) for i, j in zip(results['logits'], results['start_pos'])])
            first_logits_roc, first_logits_acc = gen_classifier_roc_5_layer(first_logits)
            classifier_results['first_logits_roc'] = first_logits_roc
            classifier_results['first_logits_acc'] = first_logits_acc

            # fully connected
            print("Classify on fully connected")
            for layer in range(results['first_fully_connected'][0].shape[0]):
                layer_roc, layer_acc = gen_classifier_roc_5_layer(np.stack([i[layer] for i in results['first_fully_connected']]))
                classifier_results[f'first_fully_connected_roc_{layer}'] = layer_roc
                classifier_results[f'first_fully_connected_acc_{layer}'] = layer_acc

            # attention
            print("Classify on attention")
            for layer in range(results['first_attention'][0].shape[0]):
                layer_roc, layer_acc = gen_classifier_roc_5_layer(np.stack([i[layer] for i in results['first_attention']]))
                classifier_results[f'first_attention_roc_{layer}'] = layer_roc
                classifier_results[f'first_attention_acc_{layer}'] = layer_acc
            
            all_results[results_file] = classifier_results.copy()
        except Exception as err:
            print(err)

Classify on logits


100%|██████████| 1001/1001 [01:24<00:00, 11.86it/s]


Classify on fully connected


100%|██████████| 1001/1001 [00:22<00:00, 43.52it/s]


Classify on attention


100%|██████████| 1001/1001 [00:09<00:00, 107.58it/s]


In [5]:
for k,v in all_results.items():
    print(f"Results for {k}:")
    for kk, vv in v.items():
        print(kk, vv)

Results for results\800\gpt2_trivia_qa_start-0_end-800.pickle:
first_logits_roc 0.3630573248407643
first_logits_acc 0.98125
first_fully_connected_roc_0 0.4288747346072187
first_fully_connected_acc_0 0.98125
first_fully_connected_roc_1 0.5732484076433121
first_fully_connected_acc_1 0.975
first_fully_connected_roc_2 0.5966029723991507
first_fully_connected_acc_2 0.98125
first_fully_connected_roc_3 0.6815286624203821
first_fully_connected_acc_3 0.98125
first_fully_connected_roc_4 0.6284501061571125
first_fully_connected_acc_4 0.975
first_fully_connected_roc_5 0.5626326963906582
first_fully_connected_acc_5 0.975
first_fully_connected_roc_6 0.5138004246284501
first_fully_connected_acc_6 0.975
first_fully_connected_roc_7 0.643312101910828
first_fully_connected_acc_7 0.98125
first_fully_connected_roc_8 0.6900212314225053
first_fully_connected_acc_8 0.98125
first_fully_connected_roc_9 0.692144373673036
first_fully_connected_acc_9 0.98125
first_fully_connected_roc_10 0.6687898089171974
first_fu

AUROC of 5-layer NN on GPT-2 and TriviaQA (800 samples):
- Softmax probability: 0.36
- Fully-connected activation: 0.75 (layer 11)
- Self-attention score: 0.74 (layer 6) | 0.7 (layer 11)